In [6]:
#import libraries

import os
import warnings
import pandas as pd

from dotenv import load_dotenv
from groq import Groq

warnings.filterwarnings("ignore")

#import libraries

from gliner import GLiNER
from qdrant_client import QdrantClient
from neo4j import GraphDatabase
from fastembed import TextEmbedding

In [7]:
load_dotenv()

True

In [8]:
#initialize groq client

client=Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

print("groq initialized")

groq initialized


In [9]:
#real healthcare query

user_query="""

I have chest pain,dizziness and breathing difficulty

"""

In [10]:
#qdrant configuration

QDRANT_URL=os.getenv(
    "QDRANT_URL"
)

QDRANT_API_KEY=os.getenv(
    "QDRANT_API_KEY"
)

COLLECTION_NAME="healthcare_conversations"

In [11]:
#neo4j configuration

NEO4J_URI=os.getenv(
    "NEO4J_URI"
)

NEO4J_USERNAME=os.getenv(
    "NEO4J_USERNAME"
)

NEO4J_PASSWORD=os.getenv(
    "NEO4J_PASSWORD"
)

In [12]:
#initialize qdrant client

qdrant_client=QdrantClient(

    url=QDRANT_URL,

    api_key=QDRANT_API_KEY

)

print("qdrant connected")

qdrant connected


In [13]:
#initialize neo4j driver

neo4j_driver=GraphDatabase.driver(

    NEO4J_URI,

    auth=(

        NEO4J_USERNAME,
        NEO4J_PASSWORD

    )

)

print("neo4j connected")

neo4j connected


In [14]:
#initialize embedding model

embedding_model=TextEmbedding(

    model_name="BAAI/bge-small-en-v1.5"

)

print("embedding model loaded")

embedding model loaded


In [15]:
#initialize gliner model

gliner_model=GLiNER.from_pretrained(
    "urchade/gliner_medium-v2.1"
)

print("gliner loaded")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

gliner loaded


In [16]:
#define medical labels

labels=[
    "symptom",
    "disease",
    "medication",
    "medical_condition",
    "body_part"
]

In [17]:
#extract entities

def extract_medical_entities(query):

    entities=gliner_model.predict_entities(

        query,

        labels

    )

    extracted=[]

    for entity in entities:

        extracted.append({

            "text":entity["text"].lower(),

            "label":entity["label"]

        })

    return extracted

In [18]:
#generate embeddings

def generate_embedding(text):

    embedding=list(

        embedding_model.embed(
            [text]
        )

    )[0]

    return embedding

In [19]:
#qdrant retrieval

def qdrant_retrieval(query):

    query_vector=generate_embedding(
        query
    )

    results=qdrant_client.query_points(

        collection_name=COLLECTION_NAME,

        query=query_vector,

        limit=5

    ).points

    semantic_contexts=[]

    for result in results:

        semantic_contexts.append({

            "score":result.score,

            "context":result.payload[
                "retrieval_text"
            ]

        })

    return semantic_contexts

In [20]:
#neo4j graph retrieval

def neo4j_graph_retrieval(entities):

    graph_contexts=[]

    with neo4j_driver.session() as session:

        for entity in entities:

            result=session.run("""

            MATCH (a)-[r]->(b)

            WHERE toLower(a.name)=toLower($entity)

            RETURN a.name AS source,
                   type(r) AS relationship,
                   b.name AS target

            LIMIT 10

            """,

            entity=entity["text"]

            )

            for record in result:

                graph_contexts.append({

                    "source":record["source"],

                    "relationship":record["relationship"],

                    "target":record["target"]

                })

    return graph_contexts

In [21]:
#hybrid retrieval pipeline

def hybrid_retrieval_pipeline(user_query):

    extracted_entities=extract_medical_entities(
        user_query
    )

    semantic_contexts=qdrant_retrieval(
        user_query
    )

    graph_contexts=neo4j_graph_retrieval(
        extracted_entities
    )

    return {

        "query":user_query,

        "entities":extracted_entities,

        "semantic_contexts":semantic_contexts,

        "graph_contexts":graph_contexts

    }

In [22]:
hybrid_result=hybrid_retrieval_pipeline(
    user_query
)

In [23]:
#extract semantic contexts

semantic_contexts=[]

for item in hybrid_result[
    "semantic_contexts"
]:

    semantic_contexts.append(

        item["context"]

    )

print(
    len(semantic_contexts)
)

5


In [24]:
#extract graph relationships

graph_relationships=[]

for item in hybrid_result[
    "graph_contexts"
]:

    graph_relationships.append({

        "source":item["source"],
        "relationship":item["relationship"],
        "target":item["target"]

    })

print(
    len(graph_relationships)
)

26


In [26]:
#load validated relationship datasets

very_high_df=pd.read_csv(
    r"C:\Users\arsha\OneDrive\Desktop\Healthcare Retrieval using qql\Data\very_high_relationships.csv"
)

high_df=pd.read_csv(
    r"C:\Users\arsha\OneDrive\Desktop\Healthcare Retrieval using qql\Data\high_relationships.csv"
)

medium_df=pd.read_csv(
    r"C:\Users\arsha\OneDrive\Desktop\Healthcare Retrieval using qql\Data\medium_relationships.csv"
)
low_df=pd.read_csv(
    r"C:\Users\arsha\OneDrive\Desktop\Healthcare Retrieval using qql\Data\low_relationships.csv"
)
very_low_df=pd.read_csv(
    r"C:\Users\arsha\OneDrive\Desktop\Healthcare Retrieval using qql\Data\very_low_relationships.csv"
)

print("validated datasets loaded")

validated datasets loaded


In [27]:
#create trusted graph context

trusted_graph_context=[]

In [28]:
#very high confidence relationships

for _,row in very_high_df.iterrows():

    trusted_graph_context.append(

        f"""

        {row['source']}
        {row['relationship']}
        {row['target']}

        Clinical Confidence:
        VERY_HIGH

        """

    )

In [29]:
#high confidence relationships

for _,row in high_df.iterrows():

    trusted_graph_context.append(

        f"""

        {row['source']}
        {row['relationship']}
        {row['target']}

        Clinical Confidence:
        HIGH

        """

    )

In [30]:
#medium confidence relationships

for _,row in medium_df.iterrows():

    trusted_graph_context.append(

        f"""

        {row['source']}
        {row['relationship']}
        {row['target']}

        Clinical Confidence:
        MEDIUM

        """

    )

In [31]:
#print trusted graph context count

print(
    len(trusted_graph_context)
)

120


In [32]:
#routing agent

def routing_agent(query):

    query=query.lower()

    selected_agents=[]

    #cardiology routing

    if any(symptom in query for symptom in [

        "chest pain",
        "heart",
        "cardiac",
        "blood pressure",
        "palpitations"

    ]):

        selected_agents.append(
            "cardiology_agent"
        )

    #neurology routing

    if any(symptom in query for symptom in [

        "dizziness",
        "vertigo",
        "headache",
        "migraine",
        "seizure"

    ]):

        selected_agents.append(
            "neurology_agent"
        )

    #pulmonology routing

    if any(symptom in query for symptom in [

        "breathing difficulty",
        "cough",
        "asthma",
        "bronchitis",
        "copd"

    ]):

        selected_agents.append(
            "pulmonology_agent"
        )

    #fallback routing

    if len(selected_agents)==0:

        selected_agents.append(
            "general_physician_agent"
        )

    return selected_agents

In [33]:
#cardiology specialist agent

def cardiology_agent(

    user_query,
    graph_context,
    semantic_context

):

    prompt=f"""

    You are an expert cardiology specialist.

    User Query:

    {user_query}

    Trusted Graph Context:

    {graph_context}

    Semantic Medical Context:

    {semantic_context}

    Tasks:

    1. Focus only on cardiac reasoning.
    2. Analyze possible heart-related concerns.
    3. Avoid hallucinations.
    4. Mention possible risks carefully.
    5. Do not provide final diagnosis.

    """

    response=client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[

            {
                "role":"system",
                "content":"You are a cardiology specialist AI."
            },

            {
                "role":"user",
                "content":prompt
            }

        ],

        temperature=0.2

    )

    return response.choices[
        0
    ].message.content

In [34]:
#neurology specialist agent

def neurology_agent(

    user_query,
    graph_context,
    semantic_context

):

    prompt=f"""

    You are an expert neurology specialist.

    User Query:

    {user_query}

    Trusted Graph Context:

    {graph_context}

    Semantic Medical Context:

    {semantic_context}

    Tasks:

    1. Focus on neurological reasoning.
    2. Analyze dizziness and vestibular symptoms.
    3. Avoid hallucinations.
    4. Avoid weak assumptions.
    5. Do not provide final diagnosis.

    """

    response=client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[

            {
                "role":"system",
                "content":"You are a neurology specialist AI."
            },

            {
                "role":"user",
                "content":prompt
            }

        ],

        temperature=0.2

    )

    return response.choices[
        0
    ].message.content

In [35]:
#pulmonology specialist agent

def pulmonology_agent(

    user_query,
    graph_context,
    semantic_context

):

    prompt=f"""

    You are an expert pulmonology specialist.

    User Query:

    {user_query}

    Trusted Graph Context:

    {graph_context}

    Semantic Medical Context:

    {semantic_context}

    Tasks:

    1. Focus on respiratory reasoning.
    2. Analyze breathing-related symptoms.
    3. Avoid hallucinations.
    4. Mention respiratory concerns carefully.
    5. Do not provide final diagnosis.

    """

    response=client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[

            {
                "role":"system",
                "content":"You are a pulmonology specialist AI."
            },

            {
                "role":"user",
                "content":prompt
            }

        ],

        temperature=0.2

    )

    return response.choices[
        0
    ].message.content

In [36]:
#general physician fallback agent

def general_physician_agent(

    user_query,
    graph_context,
    semantic_context

):

    prompt=f"""

    You are an experienced general physician.

    User Query:

    {user_query}

    Trusted Graph Context:

    {graph_context}

    Semantic Medical Context:

    {semantic_context}

    Tasks:

    1. Give general medical reasoning.
    2. Avoid hallucinations.
    3. Mention when specialist consultation may help.
    4. Do not provide final diagnosis.

    """

    response=client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[

            {
                "role":"system",
                "content":"You are a general physician AI."
            },

            {
                "role":"user",
                "content":prompt
            }

        ],

        temperature=0.2

    )

    return response.choices[
        0
    ].message.content

In [37]:
#select specialist agents

selected_agents=routing_agent(
    user_query
)

print(selected_agents)

['cardiology_agent', 'neurology_agent', 'pulmonology_agent']


In [38]:
#store specialist outputs

agent_outputs={}

In [ ]:
#execute cardiology agent

if "cardiology_agent" in selected_agents:

    cardiology_output=cardiology_agent(

        user_query,

        trusted_graph_context,

        semantic_contexts

    )

    agent_outputs[
        "cardiology_agent"
    ]=cardiology_output

In [ ]:
#execute neurology agent

if "neurology_agent" in selected_agents:

    neurology_output=neurology_agent(

        user_query,

        trusted_graph_context,

        semantic_contexts

    )

    agent_outputs[
        "neurology_agent"
    ]=neurology_output

In [ ]:
#execute pulmonology agent

if "pulmonology_agent" in selected_agents:

    pulmonology_output=pulmonology_agent(

        user_query,

        trusted_graph_context,

        semantic_contexts

    )

    agent_outputs[
        "pulmonology_agent"
    ]=pulmonology_output

In [42]:
#print specialist outputs

for agent_name,output in agent_outputs.items():

    print("\n")
    print("="*80)

    print(agent_name.upper())

    print("="*80)

    print(output)

    print("\n")

In [43]:
#final response synthesizer agent

def final_response_agent(

    user_query,
    specialist_outputs

):

    prompt=f"""

    You are a senior healthcare orchestration AI.

    User Query:

    {user_query}

    Specialist Outputs:

    {specialist_outputs}

    Tasks:

    1. Combine specialist reasoning carefully.
    2. Avoid hallucinations.
    3. Avoid final diagnosis.
    4. Mention possible concerns carefully.
    5. Suggest professional consultation if needed.
    6. Generate safe and grounded healthcare response.

    """

    response=client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[

            {
                "role":"system",
                "content":"You are a healthcare orchestration AI."
            },

            {
                "role":"user",
                "content":prompt
            }

        ],

        temperature=0.2

    )

    return response.choices[
        0
    ].message.content

In [44]:
#generate final healthcare response

final_response=final_response_agent(

    user_query,

    agent_outputs

)

In [45]:
#print final healthcare response

print("\nFINAL HEALTHCARE RESPONSE\n")

print(final_response)


FINAL HEALTHCARE RESPONSE

I'm here to help you with your symptoms. It's essential to take your concerns seriously and provide you with a thoughtful response. Based on your symptoms of chest pain, dizziness, and breathing difficulty, there are several possible concerns that may need to be considered.

Chest pain can be a symptom of various conditions, including cardiac issues, respiratory problems, or musculoskeletal concerns. Dizziness can be related to cardiovascular, neurological, or inner ear problems. Breathing difficulty can be a sign of respiratory or cardiac conditions.

Some possible concerns that may need to be evaluated include:

* Cardiac issues, such as a heart attack or angina
* Respiratory problems, like asthma, chronic obstructive pulmonary disease (COPD), or pneumonia
* Pulmonary embolism or other blood clotting disorders
* Anxiety or panic attacks
* Other conditions that may cause similar symptoms

It's crucial to consult a healthcare professional for a thorough eval